# Optimizing Apache Spark

{doc}`Spark and Databricks <01_spark_and_databricks>` showed how to find out
what a query did, and
{doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`
how tables are stored. This notebook puts both to work on the question
every data engineer eventually faces: **how do I make this faster and
cheaper?**

Almost all of a Spark query's time goes to a few things: reading data from
storage, moving it between machines in shuffles, and waiting for the slowest
task of a stage. The techniques below are ordered by how much they usually
help, starting with the biggest win: reading less data. They build on the
concepts in
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`.

The examples use `samples.tpch.lineitem`, a table of 30 million order lines
from the TPC-H benchmark, which is large enough for layout to matter.

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

for table in ["optimize03_lineitem", "optimize03_lineitem_clustered"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}")

lineitem = spark.read.table("samples.tpch.lineitem")
lineitem.count()

29999795

## Read Less: Column Pruning and Predicate Pushdown

Delta tables store data in Parquet, a **columnar** format: each column of a
file is stored separately, so a query that needs three columns reads only
those three. Spark also pushes filters down into the scan, so rows that do
not match are discarded as early as possible. The plan shows both:

In [2]:
(
    lineitem
    .filter(F.col("l_shipdate") == "1995-01-01")
    .select("l_orderkey", "l_quantity", "l_extendedprice")
    .explain()
)

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [l_orderkey#61709L, l_quantity#61713, l_extendedprice#61714]
      +- PhotonScan parquet samples.tpch.lineitem[l_orderkey#61709L,l_quantity#61713,l_extendedprice#61714,l_shipdate#61719] DataFilters: [isnotnull(l_shipdate#61719), (l_shipdate#61719 = 1995-01-01)], DictionaryFilters: [(l_shipdate#61719 = 1995-01-01)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-east-2-uc-metastore-bucket/metastore/b..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<l_orderkey:bigint,l_quantity:decimal(18,2),l_extendedprice:decimal(18,2),l_shipdate:date>, RequiredDataFilters: [isnotnull(l_shipdate#61719), (l_shipdate#61719 = 1995-01-01)]


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing = lineitem
  partial = 
  full    = 
Corrective actions: consider running the following comma

In the `PhotonScan` line, `ReadSchema` lists only the four columns the query
uses, out of sixteen, and `DataFilters` contains the `l_shipdate` condition.
You get this for free as long as you do not defeat it: select the columns
you need early rather than carrying `*` through a pipeline, and filter on
plain columns. A filter on a function of a column, such as
`year(l_shipdate) = 1995`, can still be pushed into the scan, but it cannot
use the file statistics described next as well as a direct range condition
like `l_shipdate BETWEEN '1995-01-01' AND '1995-12-31'` can.

## Read Less: Data Skipping

Pruning columns and filtering rows still means opening every file. Delta
goes further. For each data file, the transaction log stores the minimum and
maximum value of each column, and Spark uses these statistics to **skip
files** that cannot contain matching rows, without opening them.

How many files a filter can skip depends on how the rows are laid out. To
measure it, the helper below computes the minimum and maximum of a column
for every file of a table, using the `_metadata.file_path` column, and
counts the files whose range contains the value: those are the files Spark
has to read.

In [3]:
def files_to_read(table: str, column: str, value: str) -> tuple[int, int]:
    """Return (total files, files whose min/max range contains `value`)."""
    per_file = (
        spark.read.table(table)
        .groupBy(F.col("_metadata.file_path").alias("file"))
        .agg(F.min(column).alias("min_value"), F.max(column).alias("max_value"))
    )
    total = per_file.count()
    must_read = per_file.filter((F.col("min_value") <= value) & (F.col("max_value") >= value)).count()
    return total, must_read

Make a copy of `lineitem` in the course schema. The copy keeps the order of
the source, which is sorted by order key, not by date:

In [4]:
spark.sql("""
    CREATE TABLE workspace.default.optimize03_lineitem
    AS SELECT * FROM samples.tpch.lineitem
""")

files_to_read("workspace.default.optimize03_lineitem", "l_shipdate", "1995-01-01")

(10, 10)

Every file contains shipments from the whole seven-year date range, so a
filter for one day has to read all of them. Data skipping cannot help.

Now create a second copy with **liquid clustering** on `l_shipdate`, which
was introduced in
{doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`.
A smaller target file size gives the clustering more files to spread the
dates over, which suits a table of this size:

In [5]:
spark.sql("""
    CREATE TABLE workspace.default.optimize03_lineitem_clustered
    CLUSTER BY (l_shipdate)
    TBLPROPERTIES ('delta.targetFileSize' = '32mb')
    AS SELECT * FROM samples.tpch.lineitem
""")

files_to_read("workspace.default.optimize03_lineitem_clustered", "l_shipdate", "1995-01-01")

(32, 1)

:::::{dropdown} Example: How Much Does Clustering Save?
:open:
:color: primary
:icon: pencil
:class-body: example-body

A query sums the revenue shipped on 1 January 1995. How much of each table
does it have to read?

**Unclustered copy.** All 10 files have a date range that includes
1 January 1995, so Spark reads

$$
\frac{10}{10} = 100\%
$$

of the files.

**Clustered copy.** The data is spread over 32 files, each
covering a narrow range of dates, and only 1 of them can
contain the day:

$$
\frac{1}{32} \approx 3.1\%
$$

Spark skips the other files without opening them. Both tables return the
same answer; the clustered one simply reads a small fraction of the data to
get it. The saving grows with the table: on a table of thousands of files,
a selective filter on a clustering column can skip almost all of them.
:::::

Both queries return the same result:

In [6]:
for table in ["optimize03_lineitem", "optimize03_lineitem_clustered"]:
    revenue = (
        spark.read.table(f"workspace.default.{table}")
        .filter(F.col("l_shipdate") == "1995-01-01")
        .agg(F.sum("l_extendedprice").alias("revenue"))
        .first()["revenue"]
    )
    print(table, revenue)

optimize03_lineitem 472057538.40


optimize03_lineitem_clustered 472057538.40


On data this small the difference in run time is a fraction of a second and
varies from run to run, which is why this notebook measures files rather
than seconds. To see the effect on a real workload, compare the **files
pruned** and **bytes read** in the query profile, or the `pruned_files` and
`read_bytes` columns of `system.query.history`, introduced in
{doc}`Spark and Databricks <01_spark_and_databricks>`.

### Choosing Clustering Columns

Cluster by the columns that queries filter on most, typically a date or
timestamp and perhaps one or two frequently filtered keys, such as a
customer or store ID. Up to four columns are allowed. `CLUSTER BY AUTO`
lets Databricks choose the columns from the observed query workload.

Liquid clustering replaces two older techniques you will still meet in
existing tables:

- **Partitioning** (`partitionBy` / `PARTITIONED BY`) splits a table into
  one folder per value of a column. It only works for low-cardinality
  columns, and choosing the wrong column produces huge numbers of small
  files. It cannot be changed without rewriting the table.
- **Z-ordering** (`OPTIMIZE ... ZORDER BY`) co-locates related values, but
  has to be re-run on the whole table and is not incremental.

For new tables on Databricks, use liquid clustering.

## Right-Size Files

{doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`
showed a table of 200 small files compacted into one by `OPTIMIZE`. Both
extremes hurt: too many small files waste time on opening files and reading
metadata, and too few huge files limit how many tasks can read in parallel
and how many files can be skipped. On Databricks, managed tables help you
here without any code: writes are **optimized** to produce reasonably sized
files, small files are **auto-compacted** after writes, and **predictive
optimization** runs `OPTIMIZE` when it is worth it.

What remains in your control is avoiding the patterns that create small
files in the first place: very frequent tiny appends, and `repartition`
with a large number before a write. As explained in
{ref}`Controlling Partitioning <repartition-vs-coalesce>`, use `coalesce`
to reduce the number of output files.

## Shuffle Less: Joins

After reading, shuffles are the next big cost. Joins are the most common
source of them, and the strategy Spark picks matters, as described in
{ref}`Join Strategies <join-strategies>`. Joining `lineitem` with the
50,000-row `supplier` table:

In [7]:
supplier = spark.read.table("samples.tpch.supplier")

(
    lineitem
    .join(supplier, lineitem.l_suppkey == supplier.s_suppkey)
    .groupBy("s_nationkey")
    .agg(F.sum("l_extendedprice").alias("revenue"))
    .explain()
)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[s_nationkey#63589L], functions=[finalmerge_sum(merge sum#63604, isEmpty#63605) AS sum(l_extendedprice)#63597])
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#45136]
               +- PhotonShuffleExchangeSink hashpartitioning(s_nationkey#63589L, 1024)
                  +- PhotonGroupingAgg(keys=[s_nationkey#63589L], functions=[partial_sum(l_extendedprice#63575) AS (sum#63604, isEmpty#63605)])
                     +- PhotonProject [l_extendedprice#63575, s_nationkey#63589L]
                        +- PhotonBroadcastHashJoin [l_suppkey#63572L], [s_suppkey#63586L], Inner, BuildRight, false, true, false
                           :- PhotonScan parquet samples.tpch.lineitem[l_suppkey#63572L,l_extendedprice#63575] DataFilters: [isnotnull(l_suppkey#63572L)], DictionaryFilters:

Spark chose a `PhotonBroadcastHashJoin` by itself: `supplier` is small, so a
copy is sent to every executor and `lineitem` is not shuffled for the join.
The only `hashpartitioning` exchange left is the one for the final
`groupBy` on `s_nationkey`.

Joining with the 7.5-million-row `orders` table instead:

In [8]:
orders = spark.read.table("samples.tpch.orders")

(
    lineitem
    .join(orders, lineitem.l_orderkey == orders.o_orderkey)
    .groupBy("o_orderpriority")
    .agg(F.sum("l_extendedprice").alias("revenue"))
    .explain()
)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[o_orderpriority#63665], functions=[finalmerge_sum(merge sum#63680, isEmpty#63681) AS sum(l_extendedprice)#63673])
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#45305]
               +- PhotonShuffleExchangeSink hashpartitioning(o_orderpriority#63665, 1024)
                  +- PhotonGroupingAgg(keys=[o_orderpriority#63665], functions=[partial_sum(l_extendedprice#63649) AS (sum#63680, isEmpty#63681)])
                     +- PhotonProject [l_extendedprice#63649, o_orderpriority#63665]
                        +- PhotonShuffledHashJoin [l_orderkey#63644L], [o_orderkey#63660L], Inner, BuildRight
                           :- PhotonShuffleExchangeSource false
                           :  +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#45289]
                           :

`orders` is too large to broadcast, so Spark uses a `PhotonShuffledHashJoin`,
with a `hashpartitioning` exchange on the join key for **both** sides: all
30 million `lineitem` rows and all 7.5 million `orders` rows cross the
network. Ways to make such a join cheaper:

- **Filter and project first.** Only rows and columns that survive into the
  join need shuffling. Spark pushes simple filters down automatically, but
  not filters that are only known to be selective after another join.
- **Aggregate before joining.** If you only need one row per key from a
  large table, `groupBy` it before the join, so the shuffle moves the
  aggregate rather than every detail row.
- **Broadcast when you know better.** Spark decides on broadcasting from
  size estimates. If a filtered table is much smaller than its estimate,
  `F.broadcast(df)` forces the broadcast. Keep broadcast tables comfortably
  small, at most a few hundred megabytes.
- **Keep statistics current.** `ANALYZE TABLE ... COMPUTE STATISTICS FOR ALL
  COLUMNS` gives the optimiser accurate row counts and value distributions,
  which improve join order and strategy choices. Predictive optimization
  runs it for managed tables, and the `Optimizer Statistics` section at the
  bottom of every plan above shows which tables lack statistics.

Adaptive Query Execution, covered in
{ref}`Adaptive Query Execution <adaptive-query-execution>`, corrects some
of these decisions at run time, for example by switching to a broadcast join
when one side turns out to be small after filtering. It is always on in
serverless compute.

## Handle Skew

A stage finishes when its slowest task finishes. If one key has far more
rows than the others, the task that gets that key does most of the work
while the rest of the cluster waits. To see what skew looks like, generate
two million rows where 60% share the same key:

In [9]:
events = spark.range(2_000_000).withColumn(
    "key",
    F.when(F.col("id") % 10 < 6, F.lit(0)).otherwise(F.col("id") % 1000),
)

rows_per_key = events.groupBy("key").count()

rows_per_key.agg(
    F.count("*").alias("keys"),
    F.max("count").alias("largest_key"),
    F.median("count").alias("median_key"),
).show()

+----+-----------+----------+
|keys|largest_key|median_key|
+----+-----------+----------+
| 401|    1200000|    2000.0|
+----+-----------+----------+



The largest key holds 1,200,000 rows, 600 times as many as the
median key. In a join or aggregation on `key`, one task would process
more than half of all the data.

Measuring rows per key like this, and comparing the maximum with the median,
is the quickest way to confirm skew. In the query profile or Spark UI it
shows up as a stage where one task runs far longer than the others.

**Salting** spreads a hot key over several tasks: add a random number, the
salt, to the grouping key, aggregate by key and salt, then aggregate again
by key alone. The first aggregation is spread over many tasks, and the
second only has to combine 16 partial results per key:

In [10]:
SALT_BUCKETS = 16

salted = events.withColumn("salt", (F.rand(seed=42) * SALT_BUCKETS).cast("int"))

partial = salted.groupBy("key", "salt").count()
partial.agg(F.max("count").alias("largest_group_after_salting")).show()

totals = partial.groupBy("key").agg(F.sum("count").alias("count"))
totals.filter(F.col("key") == 0).show()

+---------------------------+
|largest_group_after_salting|
+---------------------------+
|                      75701|
+---------------------------+



+---+-------+
|key|  count|
+---+-------+
|  0|1200000|
+---+-------+



The largest group in the first, expensive aggregation shrank from
1,200,000 rows to 75,701, roughly
$1/16$ of the hot key, and the second aggregation restores the exact total.

For joins, AQE's skew-join handling splits oversized partitions
automatically, so try the simpler remedies from
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`
first, and salt only when a skewed aggregation or join is still the
bottleneck.

## Let the Engine Do the Work

Some of the biggest speed-ups come from staying inside what the engine can
optimise:

- **Photon.** Databricks' vectorised engine, written in C++, runs most
  DataFrame and SQL operations several times faster than the JVM engine.
  It is always on in serverless compute. The `Photon Explanation` section of
  every plan above says whether the whole query was supported. Operations it
  does not support, such as Python UDFs, fall back to the slower path.
- **Built-in functions over Python UDFs.** A Python UDF moves every row out
  of the engine into a Python process and back. Expressing the same logic
  with `pyspark.sql.functions`, as in
  {doc}`DataFrame API Basics <../02_developing_applications_with_apache_spark/01_dataframe_api_basics>`,
  keeps it in Photon. If a UDF is unavoidable, a vectorised pandas UDF is
  much faster than a row-by-row one.
- **Few actions.** Every action re-runs its plan from the source. A
  pipeline that calls `count()` for logging between steps reads its input
  several times. Use observed metrics, shown in
  {doc}`Spark and Databricks <01_spark_and_databricks>`, instead.

### Caching

On classic compute, `df.cache()` keeps a DataFrame in executor memory so
that several actions can reuse it. Serverless compute does not support it:

In [11]:
try:
    lineitem.cache().count()
except Exception as e:
    print(type(e).__name__, "-", str(e).splitlines()[0])

AnalysisException - [NOT_SUPPORTED_WITH_SERVERLESS] PERSIST TABLE is not supported on serverless compute. SQLSTATE: 0A000


In practice this matters less than it seems. Databricks caches recently
read data files on the local disks of the compute (the **disk cache**), and
SQL warehouses reuse the results of identical queries (the **result
cache**), both automatically. Where a pipeline really reuses an expensive
intermediate result, write it to a table. That also makes the pipeline
easier to debug and restart.

## Choosing Compute

Finally, the compute itself:

- **Serverless** sizes and scales the compute for each workload, and starts
  in seconds. Serverless jobs can run in **performance-optimized** mode for
  the fastest start-up and execution, or in **standard** mode, which is
  cheaper for jobs that are not latency-sensitive.
- **Classic** clusters are sized by you. Pick memory-optimised instances
  for workloads that spill or cache, keep streaming clusters at a fixed
  size, and enable Photon.

Whatever the compute, measure before and after every change: in the query
profile, in `system.query.history`, and in the cost tables shown in
{doc}`Spark and Databricks <01_spark_and_databricks>`.

## A Checklist

1. **Read less.** Select only the needed columns, filter early on plain
   columns, and cluster large tables by the columns queries filter on.
2. **Keep files healthy.** Avoid writing many small files, and let
   predictive optimization run `OPTIMIZE`, `VACUUM` and `ANALYZE`.
3. **Shuffle less.** Count the exchanges in the plan. Broadcast small
   tables, aggregate before joining, and keep statistics current.
4. **Check for skew.** Compare the largest key with the median, and the
   slowest task with the typical one.
5. **Stay in the engine.** Prefer built-in functions to Python UDFs, and
   avoid unnecessary actions.
6. **Measure.** Use query profiles and system tables to confirm that a
   change helped.

## Cleaning Up

In [12]:
for table in ["optimize03_lineitem", "optimize03_lineitem_clustered"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}");

## Resources

- [Optimization recommendations on Databricks (Databricks)](https://docs.databricks.com/aws/en/optimizations/)
- [Use liquid clustering for tables (Databricks)](https://docs.databricks.com/aws/en/delta/clustering)
- [Data skipping for Delta Lake (Databricks)](https://docs.databricks.com/aws/en/delta/data-skipping)
- [Adaptive query execution (Databricks)](https://docs.databricks.com/aws/en/optimizations/aqe)
- [What is Photon? (Databricks)](https://docs.databricks.com/aws/en/compute/photon)
- [Performance Tuning (Apache Spark)](https://spark.apache.org/docs/latest/sql-performance-tuning.html)